# pyconfind — example walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/timodonnell/pyconfind/blob/main/examples/pyconfind_demo.ipynb)

[pyconfind](https://github.com/timodonnell/pyconfind) computes rotamer-based
side-chain **contact degrees** for protein structures — a fast, modern
reimplementation of [confind](https://grigoryanlab.org/confind/) whose output is
byte-for-byte identical to the original C++ binary.

This notebook shows how to:

1. install pyconfind and fetch the inputs it needs,
2. run an analysis through the **library API** (`pyconfind.analyze`),
3. **visualize** the results — a contact map, per-residue scores, and a 3D
   structure colored by contact degree.

It runs end-to-end on a free Colab CPU runtime.

## 1. Install

We install pyconfind with the optional `[fast]` extra (the Numba JIT backend)
plus `py3Dmol` for the 3D view. Takes ~30 s.

In [ ]:
%pip install -q "pyconfind[fast] @ git+https://github.com/timodonnell/pyconfind.git" py3Dmol pandas matplotlib

## 2. Get the inputs

pyconfind needs two things: a **structure** (PDB) and a **rotamer library**.

We download a structure straight from the RCSB PDB. Here we use
[1UBQ](https://www.rcsb.org/structure/1UBQ) (ubiquitin, 76 residues) — small
enough to be fast, large enough for an interesting contact map. Swap in any
4-letter PDB id.

In [ ]:
import urllib.request

PDB_ID = "1UBQ"  # try e.g. 1CRN, 3GB1, 1LYZ ...
pdb_path = f"{PDB_ID}.pdb"
urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.pdb", pdb_path)
print(f"downloaded {pdb_path}")

The rotamer library is the Dunbrack 2010 backbone-dependent library shipped
with the upstream confind distribution. We pull it from the upstream tarball
and extract just the `rotlibs/` directory (~65 MB download, one-time).

> Tip: for a no-download quick start you can instead clone the repo and point
> at the bundled `tests/data/mini_rotlib` — a tiny truncated library. Results
> are approximate there; the full library below reproduces the C++ reference
> exactly.

In [ ]:
import os
import tarfile
import urllib.request

ROTLIB = "confind-msl/rotlibs"
if not os.path.exists(ROTLIB):
    print("downloading rotamer library (~65 MB)...")
    urllib.request.urlretrieve(
        "https://grigoryanlab.org/confind/confind-msl.tar.gz", "confind-msl.tar.gz"
    )
    with tarfile.open("confind-msl.tar.gz") as t:
        members = [m for m in t.getmembers() if m.name.startswith("confind-msl/rotlibs/")]
        t.extractall(members=members)
print("rotamer library at", ROTLIB, "->", os.listdir(ROTLIB))

## 3. Run the analysis (library API)

One call does everything: parse the PDB, build and prune rotamers at every
position, and compute the pairwise contact degrees plus the per-residue
summaries (sum contact degree, *crowdedness*, *freedom*).

`analyze` uses the Numba backend automatically when it is installed; pass
`backend="python"` to force the pure-NumPy reference (identical results).

In [ ]:
import time

from pyconfind import analyze

t0 = time.perf_counter()
result = analyze(pdb_path, rotamer_library=ROTLIB)
print(f"analyzed {len(result.positions)} residues in {time.perf_counter() - t0:.1f} s "
      f"(first call includes one-time library load + JIT warm-up)")
print(f"found {len(result.report.contacts)} residue-residue contacts")

### What's in the result

`result.positions` is one record per residue; `result.report` holds the
contacts and per-residue arrays. Let's put the per-residue scores into a
DataFrame.

In [ ]:
import pandas as pd

rep = result.report
df = pd.DataFrame(
    {
        "chain": [p.position.chain for p in result.positions],
        "resnum": [p.position.resnum for p in result.positions],
        "resname": [p.position.resname for p in result.positions],
        "sum_contact_degree": rep.sum_contact_degree,
        "crowdedness": rep.crwdnes,
        "freedom": rep.freedom,
    }
)
df.head(10)

In [ ]:
# Strongest individual contacts
top = sorted(rep.contacts, key=lambda c: c.degree, reverse=True)[:10]
for c in top:
    pi, pj = result.positions[c.pos_i].position, result.positions[c.pos_j].position
    print(f"{pi.resname}{pi.resnum} -- {pj.resname}{pj.resnum}: {c.degree:.3f}")

## 4. Visualize

### Contact map

A residue x residue heatmap of contact degree. The band near the diagonal is
sequence-local packing; off-diagonal blocks are tertiary contacts (helix
packing, beta-sheet pairing, etc.).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = len(result.positions)
M = np.zeros((N, N))
for c in rep.contacts:
    M[c.pos_i, c.pos_j] = M[c.pos_j, c.pos_i] = c.degree

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M, cmap="magma", origin="lower")
ax.set_xlabel("residue index")
ax.set_ylabel("residue index")
ax.set_title(f"{PDB_ID} contact-degree map")
fig.colorbar(im, ax=ax, label="contact degree")
plt.tight_layout()
plt.show()

### Per-residue scores

* **sum contact degree** — total contact a residue makes (buried, well-packed
  residues score high).
* **crowdedness** — fraction of rotamers pruned by backbone clashes.
* **freedom** — how much rotameric freedom the position retains.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
x = df["resnum"]
axes[0].bar(x, df["sum_contact_degree"], color="#2563eb")
axes[0].set_ylabel("sum contact\ndegree")
axes[1].bar(x, df["crowdedness"], color="#dc2626")
axes[1].set_ylabel("crowdedness")
axes[2].bar(x, df["freedom"], color="#16a34a")
axes[2].set_ylabel("freedom")
axes[2].set_xlabel("residue number")
axes[0].set_title(f"{PDB_ID} per-residue scores")
plt.tight_layout()
plt.show()

### 3D structure colored by contact degree

We write each residue's sum contact degree into the PDB B-factor column and let
[py3Dmol](https://3dmol.csb.pitt.edu/) color the cartoon by it — buried,
well-packed residues light up.

In [ ]:
import py3Dmol

score = {
    (p.position.chain, p.position.resnum): rep.sum_contact_degree[i]
    for i, p in enumerate(result.positions)
}
vmax = float(np.nanmax(rep.sum_contact_degree))

out_lines = []
for line in open(pdb_path):
    if line.startswith(("ATOM", "HETATM")):
        chain = line[21].strip() or "_"
        try:
            resnum = int(line[22:26])
        except ValueError:
            out_lines.append(line)
            continue
        b = score.get((chain, resnum))
        if b is not None:
            line = f"{line[:60]}{b:6.2f}{line[66:]}"
    out_lines.append(line)
scored_pdb = "".join(out_lines)

view = py3Dmol.view(width=720, height=520)
view.addModel(scored_pdb, "pdb")
view.setStyle(
    {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0, "max": vmax}}}
)
view.zoomTo()
view.show()

## 5. Other outputs & options

pyconfind can emit the original confind text format (a drop-in replacement for
downstream tools) or structured JSON, and supports selection strings
(`focus=`/`pre_select=`) and a native-only mode.

In [ ]:
from pyconfind import format_confind_text, format_json

# Original confind text format (first lines)
print("\n".join(format_confind_text(result.positions, result.report).splitlines()[:6]))
print("...\n")

# Restrict the computed/output residues with an MSL selection string
focused = analyze(pdb_path, rotamer_library=ROTLIB, focus="resi 1-20")
print(f"focused run: {len(focused.report.sum_contact_degree.nonzero()[0])} residues with contacts")

# Structured JSON for pipelines
import json
payload = json.loads(format_json(result.positions, result.report))
print("JSON keys:", list(payload))

---
That's it. See the [pyconfind repo](https://github.com/timodonnell/pyconfind)
for the CLI (`pyconfind --p input.pdb --rLib rotlibs --o out.cont`), the
benchmark notes, and the validation against the C++ reference.